[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/27_deep_learning_foundations/04_pytorch_fundamentals.ipynb)

# Bài 04 — PyTorch căn bản

**🎯 Mục tiêu**
- **Tensor**: tạo, kiểu dữ liệu, thiết bị (CPU/GPU), broadcasting, liên thông NumPy.
- **Autograd**: `requires_grad`, `backward()`, gradient **cộng dồn**, `no_grad`, `detach`.
- `nn.Module`, hàm loss, optimizer và **vòng lặp huấn luyện 5 bước**; `Dataset`/`DataLoader`.
- Huấn luyện lại mạng chữ số của bài 03 bằng PyTorch; GPU; lưu/nạp mô hình; các lỗi thường gặp.

> Cần PyTorch: `pip install -e ".[dl]"` (Google Colab đã cài sẵn).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/27_deep_learning_foundations

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError as e:
    raise ImportError("Bài này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | thiết bị: {device}")

## 1. Tensor

Tensor = mảng nhiều chiều (như `np.ndarray`) + chạy được trên GPU + tham gia autograd.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.ones(2, 2)
print("a =", a, "\nshape:", a.shape, "| dtype:", a.dtype, "| device:", a.device)
print("a @ b =", a @ b)
print("broadcasting a + [10, 20] =", a + torch.tensor([10.0, 20.0]))
print("reshape:", torch.arange(6).reshape(2, 3))
print("số nguyên mặc định:", torch.tensor([1, 2]).dtype, "| số thực mặc định:", torch.tensor([1.0]).dtype)

**NumPy ↔ PyTorch**: `torch.from_numpy(x)` và `t.numpy()` **dùng chung bộ nhớ** (trên CPU) — sửa bên này, bên kia đổi theo. Chú ý NumPy mặc định `float64`, PyTorch mặc định `float32`.

In [ ]:
x_np = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(x_np)
x_np[0] = 100.0
print("tensor sau khi sửa mảng numpy:", t, "| dtype:", t.dtype)
t32 = torch.tensor(x_np, dtype=torch.float32)                           # torch.tensor() sao chép dữ liệu
x_np[1] = -1.0
print("bản sao không đổi:", t32)

## 2. Autograd

Đánh dấu `requires_grad=True` → PyTorch ghi lại đồ thị tính toán (như engine ở bài 02, nhưng trên tensor); `backward()` điền `.grad`.

In [ ]:
w = torch.tensor([2.0, -1.0], requires_grad=True)
x_in = torch.tensor([3.0, 4.0])
y_out = torch.sigmoid(w @ x_in) * (w**2).sum()
y_out.backward()
s = torch.sigmoid(w @ x_in).detach()
manual = s * (1 - s) * x_in * (w.detach() ** 2).sum() + s * 2 * w.detach()   # quy tắc tích + chuỗi
print("autograd:", w.grad, "| tính tay:", manual)

**Gradient cộng dồn:** gọi `backward()` lần nữa → `.grad` **cộng thêm**, không ghi đè. Trong vòng lặp huấn luyện phải xoá gradient mỗi bước (`optimizer.zero_grad()`):

In [ ]:
w2 = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    loss_i = 3 * w2
    loss_i.backward()
    print(f"lần {i + 1}: w2.grad = {w2.grad.item()}  ← cộng dồn")
w2.grad.zero_()
print("sau zero_():", w2.grad.item())
with torch.no_grad():                                                   # không ghi đồ thị: dùng khi đánh giá, cập nhật tham số
    z_ng = w2 * 2
print("trong no_grad, requires_grad =", z_ng.requires_grad)

## 3. Hồi quy tuyến tính "bằng tay" với autograd

Tham số là tensor `requires_grad=True`, cập nhật trong `no_grad`:

In [ ]:
g = torch.Generator().manual_seed(0)
X_lr = torch.rand(200, 1, generator=g) * 4
y_lr = 3 * X_lr.squeeze() + 2 + torch.randn(200, generator=g) * 0.3
w_lr = torch.zeros(1, requires_grad=True)
b_lr = torch.zeros(1, requires_grad=True)
for step in range(500):
    pred = X_lr.squeeze() * w_lr + b_lr
    loss_lr = ((pred - y_lr) ** 2).mean()
    loss_lr.backward()
    with torch.no_grad():
        w_lr -= 0.05 * w_lr.grad
        b_lr -= 0.05 * b_lr.grad
    w_lr.grad.zero_()
    b_lr.grad.zero_()
print(f"w = {w_lr.item():.3f} (thật 3), b = {b_lr.item():.3f} (thật 2), MSE = {loss_lr.item():.4f}")

## 4. `nn.Module`, loss, optimizer

`nn.Module` gom tham số; `nn.Linear` tự khởi tạo trọng số; optimizer cập nhật mọi `model.parameters()`. **Vòng lặp 5 bước** chuẩn:

```python
optimizer.zero_grad()          # 1. xoá gradient cũ
logits = model(xb)             # 2. forward
loss = loss_fn(logits, yb)     # 3. tính loss
loss.backward()                # 4. backprop
optimizer.step()               # 5. cập nhật tham số
```

In [ ]:
class MLP(nn.Module):
    def __init__(self, sizes=(64, 128, 64, 10)):
        super().__init__()
        layers = []
        for i in range(len(sizes) - 1):
            layers.append(nn.Linear(sizes[i], sizes[i + 1]))
            if i < len(sizes) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)                                              # trả về LOGIT — không softmax


model = MLP()
print(model)
print("số tham số:", sum(p.numel() for p in model.parameters()))

> ⚠️ `nn.CrossEntropyLoss` nhận **logit** (chưa softmax) và nhãn kiểu `long` — nó tự gộp log-softmax + NLL ổn định như bài 03. Thêm softmax trước nó là lỗi phổ biến (huấn luyện vẫn chạy nhưng kém).

## 5. `Dataset` và `DataLoader`

In [ ]:
X_dg, y_dg = load_digits(return_X_y=True)
X_tmp, X_te, y_tmp, y_te = train_test_split(X_dg, y_dg, test_size=0.2, random_state=0, stratify=y_dg)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.2, random_state=0, stratify=y_tmp)
scaler = StandardScaler().fit(X_tr)                                     # cùng cách chia và chuẩn hoá như bài 03


def to_tensors(X, y):
    return torch.tensor(scaler.transform(X), dtype=torch.float32), torch.tensor(y, dtype=torch.long)


train_ds = TensorDataset(*to_tensors(X_tr, y_tr))
val_ds, test_ds = TensorDataset(*to_tensors(X_va, y_va)), TensorDataset(*to_tensors(X_te, y_te))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(0))
xb, yb = next(iter(train_loader))
print(f"{len(train_ds)} mẫu train | một batch: x {tuple(xb.shape)} {xb.dtype}, y {tuple(yb.shape)} {yb.dtype}")

## 6. Huấn luyện mạng chữ số

In [ ]:
def evaluate(model, ds):
    model.eval()                                                        # tắt dropout/batchnorm-training (chương 28)
    with torch.no_grad():
        X, y = ds.tensors
        logits = model(X.to(device))
        loss = nn.functional.cross_entropy(logits, y.to(device)).item()
        acc = (logits.argmax(1).cpu() == y).float().mean().item()
    return loss, acc


def fit(model, loader, epochs=30, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        history.append((*evaluate(model, train_ds), *evaluate(model, val_ds)))
    return np.array(history)


torch.manual_seed(0)
model = MLP()
t0 = time.perf_counter()
hist = fit(model, train_loader, epochs=30)
print(f"30 epoch trong {time.perf_counter() - t0:.1f}s | train acc {hist[-1, 1]:.4f} | val acc {hist[-1, 3]:.4f} | "
      f"test acc {evaluate(model, test_ds)[1]:.4f}")
plt.plot(hist[:, 0], label="train loss")
plt.plot(hist[:, 2], label="val loss")
plt.legend()
plt.xlabel("epoch")
plt.show()

Kết quả tương đương mạng NumPy ở bài 03 (~97%) — nhưng không phải tự viết backward, optimizer hay vòng batch.

## 7. GPU

`.to(device)` chuyển tensor/mô hình sang GPU; mọi tensor tham gia một phép toán phải **cùng thiết bị**. Lợi thế của GPU **tăng theo kích thước** phép toán (song song hoá hàng nghìn lõi); với mô hình rất nhỏ, chi phí chuyển dữ liệu và khởi chạy kernel có thể lấn át phần tính toán.

In [ ]:
for n in [256, 2048]:
    A_cpu = torch.randn(n, n)
    t0 = time.perf_counter()
    for _ in range(10):
        A_cpu @ A_cpu
    t_cpu = (time.perf_counter() - t0) / 10
    if torch.cuda.is_available():
        A_gpu = A_cpu.to("cuda")
        A_gpu @ A_gpu
        torch.cuda.synchronize()                                        # GPU chạy bất đồng bộ — phải đồng bộ khi đo giờ
        t0 = time.perf_counter()
        for _ in range(10):
            A_gpu @ A_gpu
        torch.cuda.synchronize()
        t_gpu = (time.perf_counter() - t0) / 10
        print(f"nhân ma trận {n}×{n}: CPU {t_cpu * 1000:7.2f} ms | GPU {t_gpu * 1000:6.3f} ms")
    else:
        print(f"nhân ma trận {n}×{n}: CPU {t_cpu * 1000:7.2f} ms (không có GPU)")

## 8. Lưu, nạp và tái lập

In [ ]:
import io

buffer = io.BytesIO()                                                   # thực tế: torch.save(model.state_dict(), "model.pt")
torch.save(model.state_dict(), buffer)
buffer.seek(0)
restored = MLP()
restored.load_state_dict(torch.load(buffer, weights_only=True))
restored.to(device)
print("accuracy sau khi nạp lại:", round(evaluate(restored, test_ds)[1], 4))
print("các khoá trong state_dict:", list(model.state_dict().keys())[:4], "...")

- Lưu **`state_dict`** (tham số), không lưu cả đối tượng — bền vững hơn khi code thay đổi.
- Tái lập: `torch.manual_seed`, `generator` cho `DataLoader`, cố định seed NumPy; trên GPU một số phép toán không tất định (xem `torch.use_deterministic_algorithms`).

## 9. Lỗi thường gặp

| Triệu chứng | Nguyên nhân thường gặp |
|---|---|
| Loss không giảm / dao động mạnh | quên `optimizer.zero_grad()`; learning rate quá lớn |
| Huấn luyện chạy nhưng kém bất thường | softmax trước `CrossEntropyLoss`; quên chuẩn hoá đầu vào |
| `RuntimeError: expected ... Long` | nhãn phải là `torch.long` cho `CrossEntropyLoss` |
| `Expected all tensors to be on the same device` | quên `.to(device)` cho dữ liệu hoặc mô hình |
| Kết quả đánh giá thay đổi mỗi lần chạy | quên `model.eval()` (dropout/batch norm) |
| Hết bộ nhớ khi đánh giá | quên `torch.no_grad()` (giữ đồ thị không cần thiết) |
| `.numpy()` báo lỗi | tensor đang cần grad hoặc ở GPU → `.detach().cpu().numpy()` |

## 🧠 Kiểm tra nhanh

1. `torch.from_numpy` khác `torch.tensor` thế nào?
2. Vì sao phải `zero_grad()` mỗi bước?
3. Viết 5 bước của vòng lặp huấn luyện.
4. `nn.CrossEntropyLoss` nhận đầu vào gì?
5. `model.eval()` và `torch.no_grad()` khác nhau thế nào?

## 📌 Tóm tắt

- Tensor ≈ ndarray + GPU + autograd; chú ý dtype (float32/long) và thiết bị.
- Autograd ghi đồ thị, `backward()` cộng dồn vào `.grad`; `no_grad` khi đánh giá/cập nhật.
- `nn.Module` + loss + optimizer + DataLoader → vòng lặp 5 bước; lưu `state_dict`.

## 📚 Tài liệu tham khảo

- PyTorch Tutorials — *Learn the Basics* (pytorch.org/tutorials)
- Stevens, Antiga & Viehmann (2020) — *Deep Learning with PyTorch*
- Paszke et al. (2019) — *PyTorch: An Imperative Style, High-Performance Deep Learning Library*